# IBEX 35 Volatility: Stage 5 — VaR Backtesting and Validation

Stage 4 produced dynamic VaR from the preferred GARCH models and a fair
EWMA benchmark, but flagged an important limitation: every model was fit
**once, on the full sample**, so its "conditional volatility" already
reflects knowledge of the entire series — including future stress episodes
like COVID-2020 relative to any earlier date. That is an **in-sample**
description of the data, not a genuine forecast, and it can make a VaR model
look better-calibrated than it would have been in real time.

This notebook does two things:

1. Runs the standard **Kupiec** and **Christoffersen** backtests (both
   implemented from scratch) on that same naive, in-sample VaR — clearly
   labeled as such — mostly to set up a baseline.
2. Builds a **genuine out-of-sample backtest**: an expanding-window
   procedure that refits the model repeatedly, using only data available at
   each point in time, and forecasts one day ahead. This is the only honest
   way to evaluate whether a VaR model would actually have worked.

We then compare the two directly. As with every other notebook in this
project: results are reported exactly as they come out, including where the
"naive" and "genuine" backtests disagree — that disagreement is itself the
most important finding in this notebook.

This notebook is self-contained: it re-downloads the same ~10y of daily data
and refits the Stage 3 preferred models from scratch.


## 1. Setup

In [1]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from arch import arch_model
from scipy import stats

# We deliberately do not blanket-silence warnings: convergence/numerical
# warnings from the arch optimizer (relevant here given the ~400 rolling
# refits in Section 7) are informative and should surface. We only filter
# one specific, known-benign message: a yfinance-internal pandas deprecation
# notice unrelated to result validity.
warnings.filterwarnings("ignore", message="Timestamp.utcnow is deprecated.*")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100


## 2. Data

In [2]:
TICKERS = {"IBEX35": "^IBEX", "SP500": "^GSPC"}
END = pd.Timestamp("2026-07-01")
START = END - pd.DateOffset(years=10)

raw = yf.download(list(TICKERS.values()), start=START, end=END, auto_adjust=True, progress=False)
prices = raw["Close"].rename(columns={v: k for k, v in TICKERS.items()})
prices = prices.dropna(how="any")

returns_pct = 100 * np.log(prices / prices.shift(1)).dropna()
returns_pct.tail()


Ticker,SP500,IBEX35
Date,,
2026-06-24,-0.098342,-0.447693
2026-06-25,-0.009921,0.637996
2026-06-26,-0.047177,-0.453526
2026-06-29,1.168156,-0.195299
2026-06-30,0.788900,0.434903


## 3. The naive, in-sample VaR series

Exactly as in Stage 4: refit the Stage 3 preferred specification per index
(GJR-GARCH for IBEX 35, EGARCH for S&P 500, Student-t innovations) **once,
on the full sample**, and build both the GARCH VaR and a *fair* EWMA
benchmark — same constant mean $\mu$ and same Student-t quantile (same
fitted $\nu$) as the GARCH model, so any gap between the two reflects the
volatility estimator alone, not the tail assumption (see Stage 4, Section 6,
for the full reasoning).

**This whole section is the biased version.** We keep it deliberately, as a
labeled baseline, specifically so Section 8 can show *exactly* how much the
look-ahead bias changes the answer once we fix it.

We also drop the first `EWMA_SEED_WINDOW` (30) observations from **both**
series before backtesting — EWMA's variance recursion is seeded from those
same 30 days' sample variance, so its own estimate is least trustworthy right
at the start, and counting breaches there would unfairly penalize (or flatter)
EWMA specifically. Trimming both models to the same evaluation start keeps
the comparison apples-to-apples.


In [3]:
preferred_spec = {
    "IBEX35": dict(vol="GARCH", p=1, o=1, q=1),   # GJR-GARCH
    "SP500": dict(vol="EGARCH", p=1, o=1, q=1),   # EGARCH
}

naive_results = {}
for idx, spec in preferred_spec.items():
    am = arch_model(returns_pct[idx], mean="Constant", dist="t", **spec)
    naive_results[idx] = am.fit(disp="off")


def std_t_quantile(nu, p):
    raw_q = stats.t.ppf(p, nu)
    scale = np.sqrt((nu - 2) / nu)
    return raw_q * scale


EWMA_SEED_WINDOW = 30


def ewma_volatility(returns, lam=0.94, seed_window=EWMA_SEED_WINDOW):
    var = np.empty(len(returns))
    var[0] = returns.iloc[:seed_window].var()
    for t in range(1, len(returns)):
        var[t] = lam * var[t - 1] + (1 - lam) * returns.iloc[t - 1] ** 2
    return pd.Series(np.sqrt(var), index=returns.index)


ewma_vol = {idx: ewma_volatility(returns_pct[idx]) for idx in returns_pct.columns}

CONF_LEVELS = [0.95, 0.99]

# Both series are trimmed to a common evaluation start that skips the EWMA's
# own warm-up window (see Section 6), so the naive GARCH-vs-EWMA backtest
# isn't biased by EWMA's least-reliable early days. The genuine OOS backtest
# in Section 7 needs no such trim: its test period starts ~1,491 observations
# in, long after EWMA's seed has decayed away (lambda=0.94 has a ~11-day
# half-life).
naive_thresholds = {}  # (idx, model, conf) -> Series of return-scale thresholds
for idx in returns_pct.columns:
    res = naive_results[idx]
    mu, nu = res.params["mu"], res.params["nu"]
    sigma = res.conditional_volatility
    for conf in CONF_LEVELS:
        p = 1 - conf
        q_p = std_t_quantile(nu, p)
        naive_thresholds[(idx, "GARCH", conf)] = (mu + sigma * q_p).iloc[EWMA_SEED_WINDOW:]
        naive_thresholds[(idx, "EWMA", conf)] = (mu + ewma_vol[idx] * q_p).iloc[EWMA_SEED_WINDOW:]


## 4. Kupiec's proportion-of-failures (POF) test

A VaR model at confidence $1-p$ should be breached, on average, a fraction
$p$ of the time — no more, no less. Kupiec's test checks exactly this
**unconditional coverage** property. Let $x$ be the observed number of
breaches out of $n$ observations, and $\hat\pi = x/n$ the observed breach
rate. Under $H_0$: the true breach probability equals the model's nominal
$p$, the likelihood-ratio statistic is

$$
LR_{uc} = -2\ln\left[\frac{(1-p)^{n-x}\, p^{x}}{(1-\hat\pi)^{n-x}\, \hat\pi^{x}}\right]
$$

— the log-likelihood of the data under the *nominal* breach probability $p$
in the numerator, against the log-likelihood under the *observed* rate
$\hat\pi$ (its maximum-likelihood estimate) in the denominator. If the model
is well calibrated, $\hat\pi \approx p$ and $LR_{uc} \approx 0$; the further
$\hat\pi$ drifts from $p$ (relative to the sample size), the larger
$LR_{uc}$ grows. Under $H_0$, $LR_{uc} \sim \chi^2(1)$ asymptotically, so we
reject correct coverage when $LR_{uc}$ exceeds the $\chi^2(1)$ critical value
(equivalently, when its p-value falls below 5%).

Note what this test does **not** check: it only cares about the *total
count* of breaches, not when they happened. Ten breaches evenly spread over
ten years and ten breaches all occurring in the same month would give
identical $LR_{uc}$ — that's exactly the gap Christoffersen's test closes.


In [4]:
def kupiec_test(x, n, p):
    pi_hat = x / n

    def log_lik(prob, x, n):
        if x == 0:
            return (n - x) * np.log(1 - prob)
        if x == n:
            return x * np.log(prob)
        return (n - x) * np.log(1 - prob) + x * np.log(prob)

    lr_uc = -2 * (log_lik(p, x, n) - log_lik(pi_hat, x, n))
    p_value = 1 - stats.chi2.cdf(lr_uc, df=1)
    return lr_uc, p_value


## 5. Christoffersen's independence and conditional-coverage tests

Model breach days as a binary indicator series $I_t \in \{0, 1\}$ and treat
it as a first-order Markov chain: does knowing whether *yesterday* was a
breach change the probability that *today* is one? Count the four possible
consecutive-day transitions:

|                | $I_t = 0$ | $I_t = 1$ |
|----------------|-----------|-----------|
| $I_{t-1} = 0$  | $n_{00}$  | $n_{01}$  |
| $I_{t-1} = 1$  | $n_{10}$  | $n_{11}$  |

and the corresponding transition probabilities $\hat\pi_{01} =
n_{01}/(n_{00}+n_{01})$ (breach tomorrow given no breach today) and
$\hat\pi_{11} = n_{11}/(n_{10}+n_{11})$ (breach tomorrow given a breach
today). Under **independence**, these two should be equal (both equal to
the overall unconditional breach rate $\hat\pi$); under **clustering**,
$\hat\pi_{11} > \hat\pi_{01}$ — a breach today makes another breach tomorrow
more likely than the baseline rate would suggest. The likelihood-ratio
statistic compares a single pooled probability $\hat\pi$ against the two
separately estimated transition probabilities:

$$
LR_{ind} = -2\ln\left[\frac{(1-\hat\pi)^{n_{00}+n_{10}}\, \hat\pi^{\,n_{01}+n_{11}}}{(1-\hat\pi_{01})^{n_{00}}\, \hat\pi_{01}^{\,n_{01}}\, (1-\hat\pi_{11})^{n_{10}}\, \hat\pi_{11}^{\,n_{11}}}\right] \sim \chi^2(1)
$$

A model can pass Kupiec (right *number* of breaches) yet fail this test (the
*wrong timing* — breaches bunched together), or vice versa. The **joint
conditional-coverage test** combines both properties into a single check —
correct coverage *and* independence — by simply adding the two independent
LR statistics:

$$
LR_{cc} = LR_{uc} + LR_{ind} \sim \chi^2(2)
$$


In [5]:
def christoffersen_independence_test(breach_indicator):
    breach = np.asarray(breach_indicator).astype(int)
    n00 = n01 = n10 = n11 = 0
    for t in range(1, len(breach)):
        prev, curr = breach[t - 1], breach[t]
        if prev == 0 and curr == 0:
            n00 += 1
        elif prev == 0 and curr == 1:
            n01 += 1
        elif prev == 1 and curr == 0:
            n10 += 1
        else:
            n11 += 1

    n0_, n1_ = n00 + n01, n10 + n11
    pi01 = n01 / n0_ if n0_ > 0 else 0.0
    pi11 = n11 / n1_ if n1_ > 0 else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)

    def term(count, prob):
        return 0.0 if count == 0 else count * np.log(prob)

    log_l0 = term(n00 + n10, 1 - pi) + term(n01 + n11, pi)
    log_l1 = term(n00, 1 - pi01) + term(n01, pi01) + term(n10, 1 - pi11) + term(n11, pi11)

    lr_ind = -2 * (log_l0 - log_l1)
    p_value = 1 - stats.chi2.cdf(lr_ind, df=1)
    counts = {"n00": n00, "n01": n01, "n10": n10, "n11": n11, "pi01": pi01, "pi11": pi11}
    return lr_ind, p_value, counts


def run_backtest(returns, thresholds_by_key, n=None):
    rows = []
    for (idx, model, conf), thr in thresholds_by_key.items():
        r = returns[idx].loc[thr.index]
        p = 1 - conf
        breach = (r < thr).astype(int)
        x = int(breach.sum())
        n_obs = n if n is not None else len(r)

        lr_uc, p_uc = kupiec_test(x, n_obs, p)
        lr_ind, p_ind, counts = christoffersen_independence_test(breach.values)
        lr_cc = lr_uc + lr_ind
        p_cc = 1 - stats.chi2.cdf(lr_cc, df=2)

        rows.append({
            "Index": idx, "Model": model, "Confidence": conf,
            "n": n_obs, "Breaches": x, "Breach rate": x / n_obs, "Expected rate": p,
            "Kupiec p": p_uc, "Independence p": p_ind, "Cond. coverage p": p_cc,
            "Pass (5%)": p_cc > 0.05,
        })
    return pd.DataFrame(rows).set_index(["Index", "Model", "Confidence"]).sort_index()


## 6. Backtest the naive, in-sample VaR

Applying both tests to Section 3's full-sample-fitted series.


In [6]:
naive_results_df = run_backtest(returns_pct, naive_thresholds)
naive_results_df.round(4)


n  Breaches  Breach rate  Expected rate  Kupiec p  \
Index  Model Confidence                                                         
IBEX35 EWMA  0.95        2456       164       0.0668           0.05    0.0003   
             0.99        2456        39       0.0159           0.01    0.0070   
       GARCH 0.95        2456       149       0.0607           0.05    0.0187   
             0.99        2456        31       0.0126           0.01    0.2095   
SP500  EWMA  0.95        2456       170       0.0692           0.05    0.0000   
             0.99        2456        50       0.0204           0.01    0.0000   
       GARCH 0.95        2456       157       0.0639           0.05    0.0024   
             0.99        2456        40       0.0163           0.01    0.0041   

                         Independence p  Cond. coverage p  Pass (5%)  
Index  Model Confidence                                               
IBEX35 EWMA  0.95                0.0001            0.0000      False  
             0.99                0.0252            0.0022      False  
       GARCH 0.95                0.0513            0.0094      False  
             0.99                0.0621            0.0799       True  
SP500  EWMA  0.95                0.2059            0.0001      False  
             0.99                0.1014            0.0000      False  
       GARCH 0.95                0.7495            0.0093      False  
             0.99                0.1700            0.0063      False

**Interpretation.** Only **IBEX 35's GJR-GARCH VaR at 99%** passes
conditional coverage (31 breaches vs. 24.6 expected, Kupiec p = 0.21, CC p =
0.080). Every other cell fails at the 5% level, mostly from **too many**
breaches rather than clustering — e.g. the S&P 500 GARCH VaR at 95% (157 vs.
123 expected, Kupiec p = 0.002) and at 99% (40 vs. 25, Kupiec p = 0.004), and
the fair EWMA benchmark fails everywhere it's tested (Kupiec p ≤ 0.007 at
every cell). **Take this table with a large grain of salt: it's exactly the
in-sample result Section 1 warned about.** The model has already "seen"
every stress episode in the sample — including COVID-2020 — when it
estimated the parameters used to describe volatility *before* those episodes
happened. Section 7 redoes this properly.


## 7. A genuine out-of-sample backtest

To evaluate a VaR model honestly, it has to be tested the way it would
actually be used: fit only on data available up to a point in time, then
judged on what happens *next*, which it could not have seen. We implement
an **expanding-window** backtest:

1. **Initial estimation window**: the first 60% of the sample (≈1,491
   observations, training data running from the start of the sample through
   late June 2022). The model is fit once on this window to produce the
   first out-of-sample forecast.
2. **Refit every 5 trading days** (≈once a week), using all data available
   up to that point (an *expanding*, not rolling, window — the training set
   only grows). Refitting the GARCH model by full maximum likelihood on
   every single day is unnecessary and slow; freezing the parameters for a
   business week between refits is a standard, defensible compromise between
   fidelity and computation time (documented here explicitly, since it's a
   modeling choice, not a law of nature).
3. At each date $t$ in the test period, call `.forecast(horizon=1)` on the
   most recently fitted model to get the **1-step-ahead** conditional
   variance forecast for date $t$ — built only from information through
   $t-1$, with parameters estimated only from data through (at most) 5 days
   before $t-1$. This is a genuine forecast, not a fitted value.

This leaves a test period of **≈995 observations, from late June 2022 to the
end of the sample** — about four years. **Note an important consequence of
the 60/40 split**: the initial training window (through June 2022) *contains*
COVID-2020, so the out-of-sample test period does **not**. Any difference
between Section 6 and this section's results reflects both (a) the
look-ahead-bias fix and (b) a test period that happens to be calmer than the
full sample. Section 8 disentangles the two by comparing against the naive
model re-evaluated on this *same* test window.

**EWMA needs no refitting at all.** With $\lambda=0.94$ fixed by convention
(not estimated), the EWMA recursion $\hat\sigma_t^2 = \lambda
\hat\sigma_{t-1}^2 + (1-\lambda) r_{t-1}^2$ only ever uses information
through $t-1$ and has no parameters that could leak future information — it
was already, "by accident," free of the look-ahead bias that affected the
naive GARCH fit. For the fair-comparison mean and tail quantile, we reuse
that day's *rolling* GARCH-fitted $\mu_t$ and $\nu_t$ (not the full-sample
values), so the EWMA benchmark is genuinely out-of-sample too.


In [7]:
def rolling_garch_oos(returns, spec, initial_window, refit_every=5):
    n = len(returns)
    dates = returns.index
    oos_dates, oos_mu, oos_sigma, oos_nu = [], [], [], []
    current_fit = None
    for i, t in enumerate(range(initial_window, n)):
        if i % refit_every == 0:
            train_data = returns.iloc[:t]
            am = arch_model(train_data, mean="Constant", dist="t", **spec)
            current_fit = am.fit(disp="off")
        forecast = current_fit.forecast(horizon=1, reindex=False)
        sigma_forecast = np.sqrt(forecast.variance.values[-1, 0])
        oos_dates.append(dates[t])
        oos_mu.append(current_fit.params["mu"])
        oos_sigma.append(sigma_forecast)
        oos_nu.append(current_fit.params["nu"])
    return pd.DataFrame(
        {"mu": oos_mu, "sigma": oos_sigma, "nu": oos_nu},
        index=pd.DatetimeIndex(oos_dates, name="Date"),
    )


n_total = len(returns_pct)
initial_window = int(0.6 * n_total)
print(f"Initial window: {initial_window} obs, through {returns_pct.index[initial_window - 1].date()}")
print(f"Test period: {n_total - initial_window} obs, {returns_pct.index[initial_window].date()} to {returns_pct.index[-1].date()}")

oos_fits = {}
for idx, spec in preferred_spec.items():
    oos_fits[idx] = rolling_garch_oos(returns_pct[idx], spec, initial_window, refit_every=5)


Initial window: 1491 obs, through 2022-06-24
Test period: 995 obs, 2022-06-27 to 2026-06-30


In [8]:
oos_thresholds = {}
for idx in returns_pct.columns:
    oos = oos_fits[idx]
    ewma_sigma_oos = ewma_vol[idx].loc[oos.index]
    for conf in CONF_LEVELS:
        p = 1 - conf
        q_p = oos["nu"].apply(lambda nu: std_t_quantile(nu, p))
        oos_thresholds[(idx, "GARCH", conf)] = oos["mu"] + oos["sigma"] * q_p
        oos_thresholds[(idx, "EWMA", conf)] = oos["mu"] + ewma_sigma_oos * q_p

oos_results_df = run_backtest(returns_pct, oos_thresholds)
oos_results_df.round(4)


n  Breaches  Breach rate  Expected rate  Kupiec p  \
Index  Model Confidence                                                        
IBEX35 EWMA  0.95        995        55       0.0553           0.05    0.4524   
             0.99        995        13       0.0131           0.01    0.3534   
       GARCH 0.95        995        53       0.0533           0.05    0.6398   
             0.99        995        10       0.0101           0.01    0.9873   
SP500  EWMA  0.95        995        67       0.0673           0.05    0.0169   
             0.99        995        15       0.0151           0.01    0.1345   
       GARCH 0.95        995        57       0.0573           0.05    0.3022   
             0.99        995        17       0.0171           0.01    0.0413   

                         Independence p  Cond. coverage p  Pass (5%)  
Index  Model Confidence                                               
IBEX35 EWMA  0.95                0.0105            0.0284      False  
             0.99                0.0094            0.0222      False  
       GARCH 0.95                0.0015            0.0057      False  
             0.99                0.0028            0.0114      False  
SP500  EWMA  0.95                0.8099            0.0560       True  
             0.99                0.2196            0.1536       True  
       GARCH 0.95                0.1457            0.2038       True  
             0.99                0.2913            0.0715       True

**Interpretation.** The genuine out-of-sample results split cleanly **by
index**, and the split is sharper than anything visible in the naive
version — which is exactly why this exercise matters. **All 4 S&P 500 cells
pass conditional coverage; all 4 IBEX 35 cells fail — every single one of
them because of breach clustering, not because of the wrong breach count:**

- **S&P 500 passes across the board**: 95% (57 breaches vs. 49.75 expected,
  Kupiec p = 0.30, independence p = 0.15, CC p = 0.20) and 99% GARCH (17
  breaches vs. 9.95 expected, Kupiec p = 0.04 — the one weak spot — but
  independence p = 0.29 and CC p = 0.07, a pass, if a narrow one). EWMA
  passes too, though its 95% cell is a narrow one (CC p = 0.056).
- **IBEX 35 fails across the board, but not for the reason you'd expect.**
  Look at the Kupiec column alone: 0.45, 0.35, 0.64, and 0.99 — every single
  IBEX cell has a *good*, sometimes suspiciously good, breach *count*. The
  problem is entirely in the independence column: 0.011, 0.009, 0.0015,
  0.0028 — every IBEX cell shows **statistically significant breach
  clustering**, strong enough on its own to fail the joint conditional
  coverage test regardless of how good the coverage number looks. This is a
  real, out-of-sample-only failure mode, invisible in Section 6: a model
  (GARCH *and* EWMA alike) that gets the average frequency of bad days right
  but is measurably slow to react once a genuine regime shift is underway,
  letting breaches bunch up instead of scattering as a well-specified model
  would.

We defer the full comparison and its explanation to Section 8.


## 8. Naive in-sample vs. genuine out-of-sample: a direct comparison

To isolate *only* the look-ahead-bias effect (and not also a change in test
window), we re-evaluate the naive, full-sample-fitted GARCH model — same
fitted $\mu$, $\sigma_t$, $\nu$ as Section 3 — restricted to the **same
out-of-sample dates** used in Section 7. Any difference between this and
Section 7's genuine OOS GARCH results is attributable purely to whether the
model's parameters were allowed to "see" data from after the date being
predicted.


In [9]:
naive_same_window_rows = []
for idx in returns_pct.columns:
    oos_dates = oos_fits[idx].index
    res = naive_results[idx]
    mu, nu = res.params["mu"], res.params["nu"]
    sigma_same_window = res.conditional_volatility.loc[oos_dates]
    r = returns_pct[idx].loc[oos_dates]
    n_obs = len(oos_dates)

    for conf in CONF_LEVELS:
        p = 1 - conf
        q_p = std_t_quantile(nu, p)
        thr = mu + sigma_same_window * q_p
        breach = (r < thr).astype(int)
        x = int(breach.sum())

        lr_uc, p_uc = kupiec_test(x, n_obs, p)
        lr_ind, p_ind, _ = christoffersen_independence_test(breach.values)
        p_cc = 1 - stats.chi2.cdf(lr_uc + lr_ind, df=2)

        naive_same_window_rows.append({
            "Index": idx, "Confidence": conf, "Breaches": x, "Breach rate": x / n_obs,
            "Kupiec p": p_uc, "Independence p": p_ind, "Cond. coverage p": p_cc,
        })

naive_same_window_df = pd.DataFrame(naive_same_window_rows).set_index(["Index", "Confidence"])

oos_garch_only = oos_results_df.xs("GARCH", level="Model")[
    ["Breaches", "Breach rate", "Kupiec p", "Independence p", "Cond. coverage p"]
]

comparison = naive_same_window_df.join(oos_garch_only, lsuffix=" (naive, same window)", rsuffix=" (genuine OOS)")
comparison.round(4)


Breaches (naive, same window)  \
Index  Confidence                                  
SP500  0.95                                   64   
       0.99                                   17   
IBEX35 0.95                                   52   
       0.99                                    9   

                   Breach rate (naive, same window)  \
Index  Confidence                                     
SP500  0.95                                  0.0643   
       0.99                                  0.0171   
IBEX35 0.95                                  0.0523   
       0.99                                  0.0090   

                   Kupiec p (naive, same window)  \
Index  Confidence                                  
SP500  0.95                               0.0467   
       0.99                               0.0413   
IBEX35 0.95                               0.7452   
       0.99                               0.7583   

                   Independence p (naive, same window)  \
Index  Confidence                                        
SP500  0.95                                     0.2201   
       0.99                                     0.2913   
IBEX35 0.95                                     0.0655   
       0.99                                     0.0662   

                   Cond. coverage p (naive, same window)  \
Index  Confidence                                          
SP500  0.95                                       0.0652   
       0.99                                       0.0715   
IBEX35 0.95                                       0.1740   
       0.99                                       0.1765   

                   Breaches (genuine OOS)  Breach rate (genuine OOS)  \
Index  Confidence                                                      
SP500  0.95                            57                     0.0573   
       0.99                            17                     0.0171   
IBEX35 0.95                            53                     0.0533   
       0.99                            10                     0.0101   

                   Kupiec p (genuine OOS)  Independence p (genuine OOS)  \
Index  Confidence                                                         
SP500  0.95                        0.3022                        0.1457   
       0.99                        0.0413                        0.2913   
IBEX35 0.95                        0.6398                        0.0015   
       0.99                        0.9873                        0.0028   

                   Cond. coverage p (genuine OOS)  
Index  Confidence                                  
SP500  0.95                                0.2038  
       0.99                                0.0715  
IBEX35 0.95                                0.0057  
       0.99                                0.0114

**Interpretation.** Holding the test window fixed and changing only whether
the model "cheated," the picture is striking — and it's the IBEX 35 rows
that make the point most clearly:

- **S&P 500, 95%**: naive shows 64 breaches (Kupiec p = 0.047, CC p = 0.065
  — borderline fail); genuine OOS shows 57 breaches and passes more
  comfortably (Kupiec p = 0.302, CC p = 0.204). Here the naive fit was
  slightly *worse* than the honest one, if anything — a reminder that
  look-ahead bias doesn't mechanically make every result look better, just
  less reliable.
- **S&P 500, 99%**: naive and genuine OOS land on the *same* breach count
  (17 of 995, purely coincidentally, since the two are built from entirely
  different — full-sample vs. rolling — parameter paths), so Kupiec is
  identical (p = 0.041) and both narrowly pass conditional coverage overall
  (CC p = 0.072) thanks to clean independence.
- **IBEX 35, 95%**: this is the clearest look-ahead-bias artifact in the
  notebook. The naive fit's independence test is a **borderline pass** (52
  breaches, independence p = 0.066 — just above the 5% line) while the
  genuine out-of-sample version **fails independence outright** (53
  breaches — a similar count, Kupiec p = 0.640 — but independence p =
  0.0015, roughly 40x more extreme). The breach *count* barely moves; what
  changes entirely is *when* the breaches happen. The naive, full-sample fit
  sits right on the edge of hiding a clustering problem that the honest,
  causally-updating model reveals clearly — because the naive fit's
  parameters were informed by the future outcome of that period all along.
- **IBEX 35, 99%**: same story, more extreme. Naive independence p = 0.066
  (borderline pass, CC p = 0.177, an overall pass); genuine OOS independence
  p = 0.0028 (a clear fail, CC p = 0.011). A model that looked essentially
  validated in-sample fails conditional coverage once tested honestly — not
  because it got the number of bad days wrong (Kupiec p = 0.99, about as
  good as a coverage test can look), but because those bad days weren't
  scattered the way independence requires.

**The general pattern**: the naive, in-sample backtest is not *uniformly*
flattering — it understates risk in some cells (S&P 500 at 95%) and
overstates it in others — but for the IBEX 35 specifically, it sits right at
the edge of masking a clustering problem that becomes unambiguous the moment
the model is no longer allowed to have seen the future. That is precisely
the failure mode a real trading desk cares about most: a VaR model that's
slow to react to a live regime change is far more dangerous than one with a
slightly-off average breach count, and it's also the failure mode a naive
in-sample backtest is least equipped to catch.


## 9. Results table and Basel traffic-light context

The genuine out-of-sample results (Section 7) are the results that matter;
we use them here as the basis for the Basel **99% VaR "traffic light"**
framework, which banks use to set the regulatory capital multiplier. Over a
rolling 250 trading-day window, **0-4 exceptions is green** (model accepted
as-is), **5-9 is yellow** (capital multiplier increases, supervisory
scrutiny), and **10+ is red** (model presumed inadequate). Our out-of-sample
test period is ≈995 observations rather than exactly 250, so we scale the
observed breach *rate* to its 250-day equivalent ($\text{rate} \times 250$)
as an indicative comparison, not a literal rolling-window backtest.


In [10]:
def basel_zone(exceptions_250):
    if exceptions_250 <= 4:
        return "Green"
    if exceptions_250 <= 9:
        return "Yellow"
    return "Red"


basel_df = oos_results_df.xs(0.99, level="Confidence").reset_index()
basel_df["250-day exceptions"] = basel_df["Breach rate"] * 250
basel_df["Basel zone"] = basel_df["250-day exceptions"].apply(basel_zone)
basel_df.set_index(["Index", "Model"])[["Breaches", "Breach rate", "250-day exceptions", "Basel zone"]]


Breaches  Breach rate  250-day exceptions Basel zone
Index  Model                                                      
IBEX35 EWMA         13     0.013065            3.266332      Green
       GARCH        10     0.010050            2.512563      Green
SP500  EWMA         15     0.015075            3.768844      Green
       GARCH        17     0.017085            4.271357     Yellow

**Interpretation.** *(see Section 10 for the full synthesis)* — briefly,
scaled to a 250-day equivalent: **IBEX 35 GARCH looks essentially perfect by
this metric** (10 breaches, a 1.01% rate against a 1% target, ≈2.5 scaled
exceptions — comfortably green), and every other cell is green too except
S&P 500 GARCH, which sits at the green/yellow boundary (≈4.3 exceptions).
Taken at face value, this table would suggest the IBEX 35 model is the
*best*-calibrated of the four — which is exactly backwards from the
conditional-coverage verdict in Section 7, where IBEX 35 fails **every**
cell on independence. That contradiction is the point: **Basel's
exception-counting framework only checks the total number of breaches, the
same thing Kupiec checks — it has no mechanism at all for detecting
clustering.** A model whose breaches arrive in a tight, dangerous cluster
during a single regime shift can look flawless by this table's standards
while badly failing the more complete test. See Section 10 for why this
matters more than the usual "different statistical power" caveat.


## 10. Honest interpretation

**Summary of what passed.** Out of 8 genuine out-of-sample (index, model,
confidence) combinations, **4 cleanly pass** conditional coverage — and they
are **all 4 of the S&P 500 cells**. **All 4 IBEX 35 cells fail**, and every
one of them fails for the *same* reason: significant breach clustering, not
a miscounted breach rate. That's a cleaner, more index-specific pattern than
the naive in-sample backtest suggested, and a more useful one for a
cross-market study — the headline finding here isn't really "GARCH vs.
EWMA," it's **"validated in the S&P 500, not validated in the IBEX 35, for a
specific and identifiable reason."**

**GARCH vs. EWMA is a secondary question here — the index split dominates.**
Within the S&P 500, both pass everywhere; GARCH's weakest cell is 99%
coverage (Kupiec p = 0.041, though independence is clean enough that
conditional coverage still passes at p = 0.072), and EWMA is actually
*better*-calibrated than GARCH at that same 99% level (Kupiec p = 0.13) —
echoing Stage 4's observation that the fat-tailed correction GARCH provides
doesn't uniformly help at every confidence level. Within the IBEX 35, EWMA's
clustering is somewhat less severe than GARCH's (independence p ≈ 0.01 for
EWMA vs. ≈ 0.002-0.003 for GARCH at both confidence levels) — small
consolation, since both fail outright, but a reminder that GARCH's more
reactive, asymmetric variance updating doesn't automatically make it less
prone to a clustering problem than the far simpler EWMA.

**The clustering finding is the single most important result in this
notebook, and it is much cleaner than it first looked.** Every IBEX 35 cell
has a *good* Kupiec p-value (0.45, 0.35, 0.64, and — for GARCH at 99% — an
almost suspiciously perfect 0.99) and a *bad* independence p-value (0.011,
0.009, 0.0015, 0.0028) at the same time. That combination is diagnostic: the
model gets the long-run frequency of bad days right on average, but is
measurably slow to adapt once a genuine regime shift begins, so the (right
number of) breaches arrive in a cluster rather than scattered through time.
A risk model that clusters its failures is more dangerous than one with a
slightly elevated average breach rate, because clustered breaches are
exactly what happens during a live crisis — a string of consecutive bad days
is a capital event, not a statistical nuisance.

**On the Basel traffic-light discrepancy, this cross-check now matters more
than a routine "different power" caveat.** IBEX 35 GARCH's 99% breach rate
(1.01%, ≈2.5 scaled exceptions) is about as close to the nominal 1% as this
kind of test ever gets, and it lands comfortably in Basel's green zone — the
same cell that fails conditional coverage outright once independence is
checked (p = 0.0028). Basel's exception-counting framework, like Kupiec
alone, simply has no way to see this failure mode: it would wave through
exactly the model this notebook's own Christoffersen test flags as
dangerous. That's a substantive limitation of count-based backtesting
frameworks, not a quirk of sample size.

**What this changes about the project's earlier conclusions.** Stage 4's
message — "GARCH clearly beats a fair EWMA benchmark" — was true in-sample
but doesn't fully survive contact with a genuine out-of-sample test: the
more important out-of-sample finding turns out to be about *which market*
validates, not *which model*. The right, more defensible summary across
Stages 4-5 is: **GARCH's dynamic volatility is a genuine improvement in
*description* of the historical data for both indices, and its VaR is
adequately calibrated out-of-sample for the S&P 500 — but for the IBEX 35,
neither GARCH nor its simpler EWMA benchmark reacts fast enough to
regime shifts to pass a real conditional-coverage test, regardless of how
good their average breach counts look.**

**The practical takeaway is unchanged in spirit, and now has a sharper,
more specific edge**: even fit honestly, out-of-sample, both models leave a
real gap for the IBEX 35 — not primarily a tail-thickness problem (Stage 4's
focus) but a *reaction-speed* problem. Three natural next steps, consistent
with how the industry has actually responded to problems in this same
family: (1) lean on **Expected Shortfall** (Stage 4) rather than VaR point
estimates alone — precisely why Basel's FRTB moved capital requirements from
99% VaR to 97.5% ES — (2) consider a shorter refit cadence or a model with
faster-adapting variance dynamics for markets prone to this clustering
failure, and (3) consider an **Extreme Value Theory (EVT)** approach (e.g.
Peaks-Over-Threshold with a Generalized Pareto tail) that models the extreme
tail directly, as a natural extension beyond the scope of this project.


## 11. Summary

1. **Kupiec and Christoffersen tests implemented from scratch** and applied
   twice: once to a naive, full-sample-fitted VaR (Section 6), and once to a
   genuine expanding-window, out-of-sample backtest that refits every 5
   trading days and forecasts one day ahead (Section 7).
2. **The naive in-sample backtest sits right on the edge of hiding a real
   problem**: for the IBEX 35, its independence p-values (≈0.066) are
   borderline passes, while the genuine out-of-sample version's independence
   p-values (≈0.002-0.003) are unambiguous failures (Section 8). Look-ahead
   bias doesn't just inflate headline numbers — it can hide the specific
   failure mode that matters most.
3. **Exactly 4 of 8 genuine out-of-sample cells pass, and they are all 4 of
   the S&P 500's** — every IBEX 35 cell fails, and always for the same
   reason.
4. **The split is by index, not by model.** GARCH and EWMA both pass for
   the S&P 500 and both fail for the IBEX 35 — the choice of volatility
   model is a secondary factor next to which market is being tested.
5. **IBEX 35 shows significant breach clustering out-of-sample in every
   cell**, despite good-to-excellent breach *counts* (Kupiec p up to 0.99) —
   arguably the most important, and most cautionary, finding in this
   project: a model can look essentially perfect on the metric everyone
   checks (how many breaches) while badly failing the metric that matters
   for a live crisis (when they happen).
6. **Basel traffic light cannot see this failure mode at all** — IBEX 35
   GARCH's 99% VaR lands comfortably in Basel's green zone, the same cell
   that fails conditional coverage outright once independence is checked.
7. **Bottom line**: out-of-sample VaR validation for this project is a
   genuine, market-specific success (S&P 500) and a genuine, specific
   failure (IBEX 35) — not a uniform pass or fail. Expected Shortfall
   (Stage 4), a faster-reacting variance specification, and an EVT-based
   tail model are the natural, industry-standard responses to a clustering
   failure like the one found here.
